# Agent-Ready ML: The Endgame MCP Server

This notebook is a **walkthrough** of how LLM agents interact with Endgame through the
[Model Context Protocol (MCP)](https://modelcontextprotocol.io). It is primarily a
reference document -- most cells are markdown explanations and illustrative JSON examples,
not executable code.

By the end of this notebook you will understand:

1. What MCP is and why it matters for ML tooling
2. How to configure the Endgame MCP server for Claude Code and Claude Desktop
3. The 20 tools and 6 resources the server exposes
4. What a realistic multi-step ML workflow looks like from the agent's perspective
5. Tips for prompting LLMs effectively when Endgame is connected

---
## 1. What is MCP?

The **Model Context Protocol** is an open standard ([modelcontextprotocol.io](https://modelcontextprotocol.io))
that lets LLM-based agents discover and invoke external tools and data sources through a
uniform JSON-RPC interface. Think of it as a "USB-C port" for AI: any compliant host
(Claude Desktop, Claude Code, Cursor, etc.) can connect to any compliant server and
immediately gain new capabilities.

An MCP server exposes two kinds of primitives:

| Primitive    | Description | Cost |
|:-------------|:------------|:-----|
| **Tools**    | Functions the agent can call with arguments. Each call executes real code and returns structured JSON. | One LLM round-trip per call |
| **Resources** | Read-only data the agent can fetch for context. Analogous to GET endpoints. | Zero LLM round-trips (injected into context) |

The Endgame MCP server (`endgame.mcp`) turns the entire library into an agent-friendly
API: **20 tools** for the full ML lifecycle and **6 resources** for zero-cost catalog browsing.

---
## 2. Setup

The Endgame MCP server runs as a subprocess launched by the host application.
Configuration is a single JSON file that tells the host how to start the server.

### 2a. Claude Code

Place a `.mcp.json` file in your project root:

In [ ]:
# .mcp.json  (place in your project root for Claude Code)
#
# This is the configuration file -- not meant to be run as Python.
# Shown here so you can copy-paste it.

MCP_CONFIG_CLAUDE_CODE = """
{
  "mcpServers": {
    "endgame": {
      "command": "python",
      "args": ["-m", "endgame.mcp"],
      "env": {
        "ENDGAME_MCP_WORKDIR": "/tmp/endgame_mcp"
      }
    }
  }
}
"""

print(MCP_CONFIG_CLAUDE_CODE)

### 2b. Claude Desktop

Add the server to your Claude Desktop config (`~/Library/Application Support/Claude/claude_desktop_config.json` on macOS, or `%APPDATA%\Claude\claude_desktop_config.json` on Windows):

In [ ]:
# Claude Desktop configuration snippet

MCP_CONFIG_DESKTOP = """
{
  "mcpServers": {
    "endgame": {
      "command": "/path/to/your/venv/bin/python",
      "args": ["-m", "endgame.mcp"],
      "env": {
        "ENDGAME_MCP_WORKDIR": "/tmp/endgame_mcp"
      }
    }
  }
}
"""

print(MCP_CONFIG_DESKTOP)

**Key points:**

- The `command` should point to the Python interpreter that has `endgame` installed.
- For Claude Desktop, use the **absolute path** to the virtualenv Python (e.g. `/home/user/project/.venv/bin/python`).
- `ENDGAME_MCP_WORKDIR` controls where the server writes temporary files (visualizations, reports, exported scripts). Defaults to `/tmp/endgame_mcp`.
- The server uses **stdio** transport by default (JSON-RPC over stdin/stdout). Pass `--sse` for HTTP Server-Sent Events transport instead.

---
## 3. Architecture

```
+-----------------------+         stdio / SSE         +------------------------+
|                       |  <--- JSON-RPC messages ---> |                        |
|   LLM Host            |                              |   endgame.mcp Server   |
|   (Claude Code,       |   tools/  (20 tools)         |                        |
|    Claude Desktop,    |   resources/  (6 resources)  |   SessionManager       |
|    Cursor, etc.)      |                              |     .datasets{}        |
|                       |                              |     .models{}          |
+-----------------------+                              |     .visualizations{}  |
                                                       +------------------------+
                                                                  |
                                                       +------------------------+
                                                       |   endgame library      |
                                                       |   (sklearn, LightGBM,  |
                                                       |    XGBoost, PyTorch,   |
                                                       |    statsforecast, ...) |
                                                       +------------------------+
```

### Session Management

The server maintains a **single session** per process. Every artifact gets a short, readable ID:

| Artifact       | ID format          | Example          |
|:---------------|:-------------------|:-----------------|
| Dataset        | `ds_<8 hex chars>` | `ds_a1b2c3d4`    |
| Model          | `model_<8 hex>`    | `model_e5f6a7b8` |
| Visualization  | `viz_<8 hex>`      | `viz_c9d0e1f2`   |

These short IDs are passed between tool calls so the agent can chain operations:
`load_data` returns a `dataset_id`, which is passed to `train_model`, which returns a `model_id`,
which is passed to `evaluate_model`, and so on.

### Stdout Protection

Many Endgame modules print progress to stdout. Since the stdio transport uses stdout for
JSON-RPC messages, the server **redirects stdout to stderr** during every tool call via
`capture_stdout()`. This prevents library output from corrupting the protocol.

---
## 4. Example Workflow: German Credit Classification

Below is a realistic conversation between a user and an LLM agent that has the Endgame
MCP server connected. Each step shows the tool call the agent makes and the JSON response
it receives.

---

**User:** *Load the German Credit dataset and build the best classifier.*

---

### Step 1: Load the data

The agent calls `load_data` with the OpenML identifier for the German Credit dataset:

```json
{
  "tool": "load_data",
  "arguments": {
    "source": "openml:credit-g",
    "target_column": "class"
  }
}
```

**Response:**

```json
{
  "status": "ok",
  "dataset_id": "ds_a1b2c3d4",
  "name": "credit-g",
  "shape": [1000, 21],
  "target_column": "class",
  "task_type": "classification",
  "meta_features": {
    "n_samples": 1000,
    "n_features": 20,
    "n_numeric": 7,
    "n_categorical": 13,
    "missing_pct": 0.0,
    "n_classes": 2
  },
  "columns": ["checking_status", "duration", "credit_history", "purpose", "credit_amount", "..."]
}
```

The agent now knows: 1000 samples, 20 features (7 numeric, 13 categorical), binary
classification, no missing values.

---

### Step 2: Get model recommendations

```json
{
  "tool": "recommend_models",
  "arguments": {
    "dataset_id": "ds_a1b2c3d4",
    "time_budget": "medium",
    "top_n": 5
  }
}
```

**Response:**

```json
{
  "status": "ok",
  "dataset": "credit-g",
  "task_type": "classification",
  "n_samples": 1000,
  "time_budget": "medium",
  "recommendations": [
    {"name": "lgbm", "display_name": "LightGBM", "family": "gbdt", "fit_time": "fast", "interpretable": false},
    {"name": "xgb", "display_name": "XGBoost", "family": "gbdt", "fit_time": "fast", "interpretable": false},
    {"name": "catboost", "display_name": "CatBoost", "family": "gbdt", "fit_time": "medium", "interpretable": false},
    {"name": "ebm", "display_name": "Explainable Boosting Machine", "family": "gbdt", "fit_time": "medium", "interpretable": true},
    {"name": "rf", "display_name": "Random Forest", "family": "tree", "fit_time": "fast", "interpretable": false}
  ]
}
```

The agent sees that LightGBM is the top recommendation. It decides to train it.

---

### Step 3: Train the model

```json
{
  "tool": "train_model",
  "arguments": {
    "dataset_id": "ds_a1b2c3d4",
    "model_name": "lgbm",
    "cv_folds": 5
  }
}
```

**Response:**

```json
{
  "status": "ok",
  "model_id": "model_e5f6a7b8",
  "model_name": "lgbm",
  "display_name": "LightGBM",
  "task_type": "classification",
  "cv_folds": 5,
  "metrics": {
    "accuracy": 0.7560,
    "f1": 0.7438,
    "roc_auc": 0.7892
  },
  "fit_time": 2.34,
  "n_features": 20
}
```

5-fold CV accuracy of 0.756 with AUC 0.789. The model is now stored in the session.

---

### Step 4: Evaluate in detail

```json
{
  "tool": "evaluate_model",
  "arguments": {
    "model_id": "model_e5f6a7b8",
    "metrics": "accuracy,f1,roc_auc,balanced_accuracy,matthews_corrcoef"
  }
}
```

**Response:**

```json
{
  "status": "ok",
  "model_id": "model_e5f6a7b8",
  "model_name": "lgbm_1",
  "evaluated_on": "OOF (ds_a1b2c3d4)",
  "n_samples": 1000,
  "metrics": {
    "accuracy": 0.7560,
    "f1": 0.7438,
    "roc_auc": 0.7892,
    "balanced_accuracy": 0.6831,
    "matthews_corrcoef": 0.4102
  }
}
```

---

### Step 5: Generate a report

```json
{
  "tool": "create_report",
  "arguments": {
    "model_id": "model_e5f6a7b8"
  }
}
```

**Response:**

```json
{
  "status": "ok",
  "visualization_id": "viz_c9d0e1f2",
  "report_type": "classification",
  "html_path": "/tmp/endgame_mcp/reports/report_lgbm_1.html"
}
```

The agent tells the user: *"I've generated a full classification report with confusion
matrix, ROC curve, and calibration plot. You can open it at
`/tmp/endgame_mcp/reports/report_lgbm_1.html`."*

---

### Step 6: Export a reproducible script

```json
{
  "tool": "export_script",
  "arguments": {
    "model_id": "model_e5f6a7b8"
  }
}
```

**Response:**

```json
{
  "status": "ok",
  "script_path": "/tmp/endgame_mcp/pipeline_lgbm_1.py",
  "script": "\"\"\"\nAuto-generated pipeline script for model: lgbm_1\nModel type: lgbm\n...\n\"\"\""
}
```

The exported script is a standalone `.py` file that loads the data, preprocesses it,
trains the same LightGBM model, evaluates it, and saves it -- fully reproducible outside
of MCP.

---
## 5. Available Tools (20)

All tools return structured JSON with a `"status": "ok"` or `"status": "error"` envelope.

### Data Tools

| Tool | Description | Key Parameters |
|:-----|:------------|:---------------|
| `load_data` | Load a dataset from CSV, Parquet, URL, or OpenML (e.g. `openml:31`) | `source`, `target_column`, `name`, `sample_n` |
| `inspect_data` | Inspect a loaded dataset: summary, describe, correlations, missing, distribution, head, dtypes | `dataset_id`, `operation`, `column` |
| `split_data` | Split a dataset into train/test sets (returns two new dataset IDs) | `dataset_id`, `test_size`, `stratify` |

### Discovery Tools

| Tool | Description | Key Parameters |
|:-----|:------------|:---------------|
| `list_models` | List available models, filtered by task type, family, speed, or interpretability | `task_type`, `family`, `interpretable_only`, `fast_only` |
| `recommend_models` | Recommend models for a dataset based on meta-features and time budget | `dataset_id`, `time_budget`, `top_n` |
| `describe_model` | Get detailed info about a specific model (parameters, capabilities, speed) | `model_name` |

### Training Tools

| Tool | Description | Key Parameters |
|:-----|:------------|:---------------|
| `train_model` | Train a single model with cross-validation, returns CV metrics | `dataset_id`, `model_name`, `params`, `cv_folds`, `metric` |
| `automl` | Run a full AutoML pipeline (preprocessing, model selection, ensembling) | `dataset_id`, `preset`, `time_limit`, `interpretable_only` |
| `quick_compare` | Compare multiple models on a dataset and return a leaderboard | `dataset_id`, `preset`, `metric` |

### Evaluation Tools

| Tool | Description | Key Parameters |
|:-----|:------------|:---------------|
| `evaluate_model` | Evaluate a trained model on OOF predictions or a held-out test set | `model_id`, `dataset_id`, `metrics` |
| `explain_model` | Get feature importance or permutation importance | `model_id`, `method`, `top_n` |

### Prediction Tools

| Tool | Description | Key Parameters |
|:-----|:------------|:---------------|
| `predict` | Generate predictions for a dataset using a trained model | `model_id`, `dataset_id`, `output_path`, `include_probabilities` |

### Preprocessing Tools

| Tool | Description | Key Parameters |
|:-----|:------------|:---------------|
| `preprocess` | Apply a chain of preprocessing operations (impute, scale, encode, balance, select, drop) | `dataset_id`, `operations` (JSON array) |

### Visualization Tools

| Tool | Description | Key Parameters |
|:-----|:------------|:---------------|
| `create_visualization` | Create a chart (ROC, PR, confusion matrix, histogram, scatter, heatmap, etc.) as self-contained HTML | `chart_type`, `model_id`, `dataset_id`, `params` |
| `create_report` | Generate a comprehensive HTML evaluation report (classification or regression) | `model_id`, `dataset_id`, `report_type` |

### Export Tools

| Tool | Description | Key Parameters |
|:-----|:------------|:---------------|
| `export_script` | Generate a standalone Python script that reproduces the trained pipeline | `model_id`, `dataset_path`, `include_preprocessing` |
| `save_model` | Save a trained model to disk using endgame persistence | `model_id`, `path` |

### Advanced Tools

| Tool | Description | Key Parameters |
|:-----|:------------|:---------------|
| `cluster` | Cluster a dataset (kmeans, hdbscan, dbscan, agglomerative, gaussian_mixture) | `dataset_id`, `method`, `n_clusters`, `params` |
| `detect_anomalies` | Detect outliers (isolation_forest, lof, elliptic_envelope) | `dataset_id`, `method`, `contamination` |
| `forecast` | Forecast a time series (auto/arima, ets, theta, naive) | `dataset_id`, `target_column`, `horizon`, `method` |

---
## 6. Available Resources (6)

Resources are read-only data endpoints the agent can fetch for context **without** executing
any computation. They are ideal for populating the agent's context window with catalog
information before deciding which tools to call.

| Resource URI | Description | Contents |
|:-------------|:------------|:---------|
| `endgame://catalog/models` | All available models grouped by family | Name, display name, fit time, interpretability, task types, description for every registered model |
| `endgame://catalog/presets` | AutoML preset configurations | Description, time limit, CV folds, model pool, ensemble method, feature engineering flags for each preset |
| `endgame://catalog/visualizers` | Available chart types with required inputs | ML evaluation charts (ROC, PR, confusion matrix, etc.) and data exploration charts (histogram, scatter, heatmap, etc.) |
| `endgame://catalog/metrics` | Evaluation metrics for classification and regression | Metric name and description for all supported metrics (accuracy, roc_auc, rmse, r2, etc.) |
| `endgame://session/state` | Current session state | All loaded datasets, trained models, and generated visualizations with their IDs and metadata |
| `endgame://guide/examples` | Example tool-call workflows for common ML tasks | Step-by-step tool call sequences for classification, regression, AutoML, model comparison, and data exploration |

---
## 7. Tips for Best Results

When prompting an LLM that has the Endgame MCP server connected, keep these tips in mind:

### Be specific about your goal

Instead of *"do some ML on this data"*, try:

> *"Load `train.csv`, train a LightGBM classifier targeting the `Survived` column,
> evaluate it with ROC AUC and F1, then export the pipeline as a script."*

The agent maps each clause to a tool call. More specificity means fewer wasted round-trips.

### State the target column early

The `target_column` parameter in `load_data` triggers automatic task-type inference
(classification vs. regression) and meta-feature computation. If you forget it, the agent
will have to reload or guess.

### Let the agent use `recommend_models`

If you are unsure which model to use, just say *"pick the best model"* or *"recommend
models for this dataset"*. The server inspects the dataset's meta-features (size,
cardinality, missing values) and returns a ranked portfolio.

### Use presets for AutoML

The `automl` tool accepts presets that control quality/speed tradeoff:

| Preset | Time budget | Model pool | Use case |
|:-------|:-----------|:-----------|:---------|
| `fast` | Minutes | Small set of fast models | Quick iteration |
| `medium_quality` | ~10 min | Balanced pool | Default starting point |
| `good_quality` | ~30 min | Larger pool with tuning | Serious evaluation |
| `high_quality` | ~1 hour | Full pool, heavy tuning | Competition/production |
| `best_quality` | Hours | Everything, ensembling | Maximum performance |
| `interpretable` | ~10 min | EBM, linear, trees only | Regulated domains |

### Ask for visualizations

The server generates self-contained HTML files for every chart. You can say:

> *"Show me the ROC curve, a confusion matrix, and a feature importance chart."*

The agent will call `create_visualization` three times and return the file paths.

### Chain operations explicitly

For complex pipelines, describe the full chain:

> *"Load the data, impute missing values with median, encode categoricals, train XGBoost
> and LightGBM, compare their AUC, then export the winner as a script."*

The agent will call `load_data` -> `preprocess` -> `train_model` (x2) -> `evaluate_model` (x2) -> `export_script`.

---
## 8. Troubleshooting

### Categorical encoding errors

**Symptom:** `ValueError: could not convert string to float` during `train_model`.

**Cause:** The chosen model does not natively handle categorical features (e.g., SVM, KNN,
neural networks). The server auto-encodes categoricals for models where
`handles_categorical=False` in the registry, but edge cases can still occur with unusual
dtypes.

**Fix:** Call `preprocess` with `[{"type": "encode", "method": "label"}]` before training,
or switch to a model that handles categoricals natively (LightGBM, CatBoost, EBM).

---

### Timeout during training

**Symptom:** `{"status": "error", "error_type": "timeout", "message": "..."}` from `train_model`.

**Cause:** The model took longer than the built-in timeout guard (default ~5 minutes per
tool call). This often happens with deep tabular models (FT-Transformer, SAINT) on larger
datasets.

**Fix:**
- Use `sample_n` in `load_data` to subsample the dataset for prototyping.
- Switch to a faster model (`lgbm`, `xgb`, `rf`).
- Use `automl` with a `time_limit` parameter to control total training time.

---

### Missing dependency errors

**Symptom:** `{"status": "error", "error_type": "missing_dependency", "message": "...", "hint": "pip install ..."}`

**Cause:** Some tools require optional dependencies that are not installed:
- `load_data` with `openml:` source requires `openml`
- `preprocess` with `balance` requires `imbalanced-learn`
- `forecast` with `arima`/`ets`/`theta` requires `statsforecast`

**Fix:** Install the package indicated in the `hint` field. The error response always
includes the exact pip command.

---

### Server not appearing in Claude Desktop

**Symptom:** The MCP tools do not appear in the tool list.

**Fix:**
1. Verify the `command` path in your config points to a Python that has `endgame` installed.
2. Test manually: run `python -m endgame.mcp` -- it should start silently (stdio mode blocks on stdin).
3. Check the Claude Desktop logs for MCP server startup errors.
4. Restart Claude Desktop after editing the config file.

---

### Visualizations not opening

**Symptom:** The agent returns an `html_path` but the file does not exist or is empty.

**Fix:** Check that `ENDGAME_MCP_WORKDIR` points to a writable directory. The server
creates subdirectories `visualizations/` and `reports/` under this path. The default
`/tmp/endgame_mcp` should work on most systems.

---

## Summary

The Endgame MCP server bridges the gap between natural language and ML pipelines.
With 20 tools spanning the full ML lifecycle -- data loading, exploration, preprocessing,
model training, evaluation, visualization, and export -- and 6 resources for zero-cost
catalog browsing, LLM agents can build complete ML workflows through conversation.

Key takeaways:

- **Configuration is minimal**: a single `.mcp.json` file.
- **Session state is automatic**: short IDs track datasets, models, and visualizations.
- **Everything is JSON**: tools return structured responses; resources return catalog data.
- **Scripts are exportable**: the agent can produce standalone Python code at any time.

For more information:
- [Model Context Protocol specification](https://modelcontextprotocol.io)
- [Endgame documentation](https://github.com/endgame-ml/endgame)
- Source code: `endgame/mcp/`